# Classify text with embeddings

### Overview

In this notebook, you'll learn to use the embeddings produced by the Gemini API to train a model that can classify different types of newsgroup posts based on the topic.


## Setup

### Install the Google GenAI SDK

Install the Google GenAI SDK from [npm](https://www.npmjs.com/package/@google/genai). 

```bash
$ npm install @google/genai
```

### Setup your API key

You can [create](https://aistudio.google.com/app/apikey) your API key using Google AI Studio with a single click.

Remember to treat your API key like a password. Don't accidentally save it in a notebook or source file you later commit to GitHub. In this notebook we will be storing the API key in a `.env` file. You can also set it as an environment variable or use a secret manager. 

Here's how to set it up in a `.env` file:

```bash
$ touch .env
$ echo "GEMINI_API_KEY=<YOUR_API_KEY>" >> .env
```

:::{.callout-tip}

Another option is to set the API key as an environment variable. You can do this in your terminal with the following command:

```bash
$ export GEMINI_API_KEY="<YOUR_API_KEY>"
```
:::

### Load the API key

To load the API key from the `.env` file, we will use the `dotenv` package. This package loads environment variables from a `.env` file into `process.env`. 

```bash
$ npm install dotenv
```

Then, we can load the API key in our code:


In [3]:
const dotenv = require("dotenv") as typeof import("dotenv");

dotenv.config({
  path: "../.env",
});

const GEMINI_API_KEY = process.env.GEMINI_API_KEY ?? "";
if (!GEMINI_API_KEY) {
  throw new Error("GEMINI_API_KEY is not set in the environment variables");
}
console.log("GEMINI_API_KEY is set in the environment variables");


GEMINI_API_KEY is set in the environment variables


:::{.callout-note}
In our particular case the `.env` is is one directory up from the notebook, hence we need to use `../` to go up one directory. If the `.env` file is in the same directory as the notebook, you can omit it altogether. 

```
│
├── .env
└── examples
    └── Classify_text_with_embeddings.ipynb
```
:::


### Initialize SDK Client

With the new SDK, now you only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.


In [4]:
const google = require("@google/genai") as typeof import("@google/genai");

const ai = new google.GoogleGenAI({ apiKey: GEMINI_API_KEY });


### Select a model

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](../quickstarts/Get_started_thinking.ipynb) for more details and in particular learn how to switch the thiking off).


In [5]:
const tslab = require("tslab") as typeof import("tslab");

const MODEL_ID = "gemini-2.5-flash-preview-05-20";


## Prepare dataset

The [20 Newsgroups Text Dataset](https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html) contains 18,000 newsgroups posts on 20 topics divided into training and test sets. The split between the training and test datasets are based on messages posted before and after a specific date. For this tutorial, you will be using the subsets of the training and test datasets. You will preprocess and organize the data into Pandas dataframes.


In [6]:
const fs = require("fs") as typeof import("fs");
const path = require("path") as typeof import("path");
const tar = require("tar") as typeof import("tar");
const danfo = require("danfojs-node") as typeof import("danfojs-node");

// URL of the scikit-learn 20 Newsgroups dataset
const DATA_URL = "https://ndownloader.figshare.com/files/5975967";
const EXTRACT_PATH = "../assets/anomaly_detection";

async function downloadAndExtractDataset(): Promise<void> {
  if (fs.existsSync(EXTRACT_PATH)) {
    console.log("Dataset already exists. Skipping download.");
    return;
  }

  console.log("Downloading 20 Newsgroups dataset...");
  const response = await fetch(DATA_URL);
  const buffer = await response.arrayBuffer();

  console.log("Extracting dataset...");
  await fs.promises.mkdir(EXTRACT_PATH, { recursive: true });

  const zipPath = path.join(EXTRACT_PATH, "20news-bydate.tar.gz");
  fs.writeFileSync(zipPath, Buffer.from(buffer));

  await tar.x({
    file: zipPath,
    cwd: EXTRACT_PATH,
  });

  console.log("Dataset extracted.");
}

function loadTextFilesFromDir(dirPath: string): {
  data: string[];
  target: string[];
} {
  const categories = fs.readdirSync(dirPath);
  const data: string[] = [];
  const target: string[] = [];

  for (const category of categories) {
    const categoryPath = path.join(dirPath, category);
    if (fs.lstatSync(categoryPath).isDirectory()) {
      const files = fs.readdirSync(categoryPath);
      for (const file of files) {
        const filePath = path.join(categoryPath, file);
        const content = fs.readFileSync(filePath, "utf-8");
        data.push(content);
        target.push(category);
      }
    }
  }

  return { data, target };
}

await downloadAndExtractDataset();

const trainDir = path.join(EXTRACT_PATH, "20news-bydate-train");
const { data: trainData, target: trainTarget } = loadTextFilesFromDir(trainDir);
const trainDf = new danfo.DataFrame({
  data: trainData,
  target: trainTarget,
});

const testDir = path.join(EXTRACT_PATH, "20news-bydate-test");
const { data: testData, target: testTarget } = loadTextFilesFromDir(testDir);
const testDf = new danfo.DataFrame({
  data: testData,
  target: testTarget,
});


Dataset already exists. Skipping download.


In [7]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call */
const classNames = trainDf.target.unique().values as string[];
console.log("Class names:", classNames);


Class names: [
  'alt.atheism',
  'comp.graphics',
  'comp.os.ms-windows.misc',
  'comp.sys.ibm.pc.hardware',
  'comp.sys.mac.hardware',
  'comp.windows.x',
  'misc.forsale',
  'rec.autos',
  'rec.motorcycles',
  'rec.sport.baseball',
  'rec.sport.hockey',
  'sci.crypt',
  'sci.electronics',
  'sci.med',
  'sci.space',
  'soc.religion.christian',
  'talk.politics.guns',
  'talk.politics.mideast',
  'talk.politics.misc',
  'talk.religion.misc'
]


Here is an example of what a data point from the training set looks like.


In [8]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access */
const firstDoc = trainDf.loc({ rows: [0], columns: ["data"] });
const firstText = firstDoc.data.values[0] as string;

const idx = firstText.indexOf("Lines");

if (idx !== -1) {
  console.log(firstText.slice(idx));
} else {
  console.log('"Lines" not found in the first document.');
}


Lines: 290

Archive-name: atheism/resources
Alt-atheism-archive-name: resources
Last-modified: 11 December 1992
Version: 1.0

                              Atheist Resources

                      Addresses of Atheist Organizations

                                     USA

FREEDOM FROM RELIGION FOUNDATION

Darwin fish bumper stickers and assorted other atheist paraphernalia are
available from the Freedom From Religion Foundation in the US.

Write to:  FFRF, P.O. Box 750, Madison, WI 53701.
Telephone: (608) 256-8900

EVOLUTION DESIGNS

Evolution Designs sell the "Darwin fish".  It's a fish symbol, like the ones
Christians stick on their cars, but with feet and the word "Darwin" written
inside.  The deluxe moulded 3D plastic fish is $4.95 postpaid in the US.

Write to:  Evolution Designs, 7119 Laurel Canyon #4, North Hollywood,
           CA 91605.

People in the San Francisco Bay area can get Darwin Fish from Lynn Gold --
try mailing <figmo@netcom.com>.  For net people who go to Lynn d

Now you will begin preprocessing the data for this tutorial. Remove any sensitive information like names, email, or redundant parts of the text like `"From: "` and `"\nSubject: "`. Organize the information into a Pandas dataframe so it is more readable.


In [ ]:
/* eslint-disable no-control-regex, @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-argument, @typescript-eslint/no-unsafe-assignment */

import { DataFrame } from "danfojs-node";

function preprocessText(text: string): string {
  let cleaned = text;

  // Remove emails
  cleaned = cleaned.replace(/[\w.-]+@[\w.-]+/g, "");

  // Remove names (assuming your original regex was incomplete due to formatting)
  // You can customize this pattern based on what "names" means in your context
  cleaned = cleaned.replace(/^(.*?)(?=\n)/g, ""); // naive: remove first line, often name

  // Remove "From: "
  cleaned = cleaned.replace(/From: /g, "");

  // Remove "\nSubject: "
  cleaned = cleaned.replace(/\nSubject: /g, "");

  // Remove control characters
  cleaned = cleaned.replace(/[\x00-\x1F\x7F]/g, " ");

  // Truncate to 5000 characters
  if (cleaned.length > 5000) {
    cleaned = cleaned.slice(0, 5000);
  }

  return cleaned;
}

function preprocessDataframe(df: DataFrame): DataFrame {
  const preprocessedData = df.data.values.map((d: string) => preprocessText(d));
  const preprocessedDf = new DataFrame({
    text: preprocessedData as string[],
    target: df.target.values,
  });
  /* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-assignment */
  const texts = preprocessedDf.text.values as string[];
  const classNameToLabelMap: Record<string, number> = preprocessedDf.target
    .unique()
    .values.reduce((acc: Record<string, number>, className: string, index: number) => {
      acc[className] = index + 1; // Start labels from 1
      return acc;
    }, {});
  const classNames = preprocessedDf.target.values as string[];
  const labels = classNames.map((name) => classNameToLabelMap[name]);
  return new DataFrame({
    Text: texts,
    Label: labels,
    "Class Name": classNames,
  });
}

const trainDfPreprocessed = preprocessDataframe(trainDf);
const testDfPreprocessed = preprocessDataframe(testDf);
trainDfPreprocessed.head().print();


╔════════════╤═══════════════════╤═══════════════════╤═══════════════════╗
║            │ Text              │ Label             │ Class Name        ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 0          │ Alt.Atheism FAQ…  │ 1                 │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 1          │ Alt.Atheism FAQ…  │ 1                 │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 2          │ Re: Gospel Dati…  │ 1                 │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 3          │ Re: university …  │ 1                 │ alt.atheism       ║
╟────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 4          │ Re: [soc.motss,…  │ 1                 │ alt.atheism       ║
╚════════════╧═══════════════════╧═══════════════════╧═══════════════════╝



Next, you will sample some of the data by taking 100 data points in the training dataset, and dropping a few of the categories to run through this tutorial. Choose the science categories to compare.


In [10]:
import { DataFrame } from "danfojs-node";

async function sampleData(df: DataFrame, numSamples: number, classesToKeep: string[]): Promise<DataFrame> {
  const uniqueLabels = df.Label.unique().values;
  const sampledGroups = [];
  for (const label of uniqueLabels) {
    const labelGroup = df.query(df.Label.eq(label)).resetIndex();
    const groupSize = labelGroup.shape[0];
    if (groupSize > 0) {
      const sampledGroup = await labelGroup.sample(numSamples, { seed: 42 });
      sampledGroups.push(sampledGroup);
    }
  }
  const dfSampled = danfo.concat({
    dfList: sampledGroups,
    axis: 0,
  }) as DataFrame;
  const mask = dfSampled["Class Name"].values.map((name: string) =>
    classesToKeep.some((c: string) => name.includes(c))
  );
  const dfFiltered = dfSampled.query(mask).resetIndex();

  const classNames = dfFiltered["Class Name"].unique().values;
  const classToCode: Record<string, number> = {};
  classNames.forEach((name: string, i: number) => {
    classToCode[name] = i;
  });

  const encodedLabel = dfFiltered["Class Name"].values.map((val: string) => classToCode[val]);
  dfFiltered.addColumn("Encoded Label", encodedLabel, { inplace: true });

  return dfFiltered.resetIndex();
}


In [11]:
const TRAIN_NUM_SAMPLES = 100;
const TEST_NUM_SAMPLES = 25;
const CLASSES_TO_KEEP = "sci";
const dfTrainFinal = await sampleData(trainDfPreprocessed, TRAIN_NUM_SAMPLES, [CLASSES_TO_KEEP]);
const dfTestFinal = await sampleData(testDfPreprocessed, TEST_NUM_SAMPLES, [CLASSES_TO_KEEP]);


In [12]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call */

import { DataFrame } from "danfojs-node";

const trainValueCounts = dfTrainFinal["Class Name"].valueCounts() as DataFrame;
trainValueCounts.print();


╔═════════════════╤═════╗
║ sci.crypt       │ 100 ║
╟─────────────────┼─────╢
║ sci.electronics │ 100 ║
╟─────────────────┼─────╢
║ sci.med         │ 100 ║
╟─────────────────┼─────╢
║ sci.space       │ 100 ║
╚═════════════════╧═════╝



In [13]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call */

import { DataFrame } from "danfojs-node";

const testValueCounts = dfTestFinal["Class Name"].valueCounts() as DataFrame;
testValueCounts.print();


╔═════════════════╤════╗
║ sci.crypt       │ 25 ║
╟─────────────────┼────╢
║ sci.electronics │ 25 ║
╟─────────────────┼────╢
║ sci.med         │ 25 ║
╟─────────────────┼────╢
║ sci.space       │ 25 ║
╚═════════════════╧════╝



## Create the embeddings

In this section, you will see how to generate embeddings for a piece of text using the embeddings from the Gemini API. To learn more about embeddings, visit the [embeddings guide](https://ai.google.dev/docs/embeddings_guide).

:::{.callout-note}

Embeddings are computed one at a time, large sample sizes can take a long time!

:::


### API changes to Embeddings

For the recent embeddings model, there is a task type parameter and the optional title (only valid with `task_type=RETRIEVAL_DOCUMENT`).

These parameters apply only to the recent embeddings models. The task types are:

| Task Type             | Description                                                                  |
| --------------------- | ---------------------------------------------------------------------------- |
| `RETRIEVAL_QUERY`     | Specifies the given text is a query in a search/retrieval setting.           |
| `RETRIEVAL_DOCUMENT`  | Specifies the given text is a document in a search/retrieval setting.        |
| `SEMANTIC_SIMILARITY` | Specifies the given text will be used for Semantic Textual Similarity (STS). |
| `CLASSIFICATION`      | Specifies that the embeddings will be used for classification.               |
| `CLUSTERING`          | Specifies that the embeddings will be used for clustering.                   |


In [14]:
/* eslint-disable @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-assignment */
import { DataFrame } from "danfojs-node";

const EMBEDDING_MODEL_ID = "models/text-embedding-004";
const BATCH_SIZE = 100;

async function addEmbeddings(df: DataFrame, textColumnName = "Text"): Promise<DataFrame> {
  const embeddings: number[][] = [];
  const display = tslab.newDisplay();
  display.text("Progress: 0%");

  for (let i = 0; i < df.shape[0]; i += BATCH_SIZE) {
    const batch = df[textColumnName].values.slice(i, i + BATCH_SIZE);
    const embeddingResponse = await ai.models.embedContent({
      model: EMBEDDING_MODEL_ID,
      contents: batch,
      config: {
        taskType: "CLASSIFICATION",
      },
    });
    const batchEmbeddings = embeddingResponse.embeddings?.map((e) => e.values ?? []) ?? [];
    embeddings.push(...batchEmbeddings);
    display.text(`Progress: ${Math.min(100, ((i + BATCH_SIZE) / df.shape[0]) * 100).toFixed(2)}%`);
  }

  const dfCopy = df.copy();
  dfCopy.addColumn("Embedding", new danfo.Series(embeddings), { inplace: true });
  return dfCopy;
}


In [15]:
const dfTrainWithEmbeddings = await addEmbeddings(dfTrainFinal);


Progress: 100.00%

In [16]:
const dfTestWithEmbeddings = await addEmbeddings(dfTestFinal);


Progress: 100.00%

In [17]:
dfTrainWithEmbeddings.head().print();


╔════════════╤═══════════════════╤═══════════════════╤═══════════════════╤═══════════════════╤═══════════════════╗
║            │ Text              │ Label             │ Class Name        │ Encoded Label     │ Embedding         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 0          │ Re: Re-inventin…  │ 12                │ sci.crypt         │ 0                 │ -0.0029816378,0…  ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 1          │ Re: Source of r…  │ 12                │ sci.crypt         │ 0                 │ -0.020604927,0.…  ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ 2          │ Re: White House…  │ 12                │ sci.crypt         │ 0                 │ 0.00362508,0.01…  ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼──────

## Build a simple classification model

Here you will define a simple model with one hidden layer and a single class probability output. The prediction will correspond to the probability of a piece of text being a particular class of news. When you build your model, Keras will automatically shuffle the data points.


In [ ]:
const tf = require("@tensorflow/tfjs-node") as typeof import("@tensorflow/tfjs-node");


In [ ]:
import { LayersModel } from "@tensorflow/tfjs-node";

function buildClassificationModel(inputSize: number, numClasses: number): LayersModel {
  const model = tf.sequential();

  model.add(
    tf.layers.dense({
      inputShape: [inputSize],
      units: inputSize,
      activation: "relu",
    })
  );

  model.add(
    tf.layers.dense({
      units: numClasses,
      activation: "softmax",
    })
  );

  return model;
}


In [20]:
const embeddingSize = (dfTestWithEmbeddings.Embedding.values[0] as string).split(",").length;
const numClasses = dfTrainWithEmbeddings["Class Name"].unique().values.length;
console.log("Embedding size:", embeddingSize);
console.log("Number of classes:", numClasses);


Embedding size: 768
Number of classes: 4


In [21]:
const classifier = buildClassificationModel(embeddingSize, numClasses);
classifier.summary();


__________________________________________________________________________________________
Layer (type)                Input Shape               Output shape              Param #   
dense_Dense1 (Dense)        [[null,768]]              [null,768]                590592    
__________________________________________________________________________________________
dense_Dense2 (Dense)        [[null,768]]              [null,4]                  3076      
Total params: 593668
Trainable params: 593668
Non-trainable params: 0
__________________________________________________________________________________________


In [22]:
classifier.compile({
  loss: "sparseCategoricalCrossentropy",
  optimizer: tf.train.adam(0.001),
  metrics: ["accuracy"],
});


## Train the model to classify newsgroups

Finally, you can train a simple model. Use a small number of epochs to avoid overfitting. The first epoch takes much longer than the rest, because the embeddings need to be computed only once.


In [ ]:
import { Tensor } from "@tensorflow/tfjs-node";

const NUM_EPOCHS = 25;
const BATCH_SIZE = 32;

const yTrain = tf.tensor1d(dfTrainWithEmbeddings["Encoded Label"].values as number[]);
const xTrain = tf.tensor2d(
  dfTrainWithEmbeddings.Embedding.values.map((e: string) =>
    e.split(",").map((v: string) => parseFloat(v))
  ) as number[][]
);
const yVal = tf.tensor1d(dfTestWithEmbeddings["Encoded Label"].values as number[]);
const xVal = tf.tensor2d(
  dfTestWithEmbeddings.Embedding.values.map((e: string) => e.split(",").map((v: string) => parseFloat(v))) as number[][]
);

let bestAcc = 0;
let bestWeights: Tensor[] = [];

const restoreBestWeightsCallback = new tf.CustomCallback({
  // eslint-disable-next-line @typescript-eslint/require-await
  onEpochEnd: async (epoch, logs) => {
    const acc = logs?.val_acc ?? 0;
    if (acc > bestAcc) {
      bestAcc = acc;
      bestWeights = classifier.getWeights().map((w) => w.clone());
    }
  },
  // eslint-disable-next-line @typescript-eslint/require-await
  onTrainEnd: async () => {
    if (bestWeights.length > 0) {
      classifier.setWeights(bestWeights);
    }
  },
});

const earlyStopping = tf.callbacks.earlyStopping({
  monitor: "val_acc",
  patience: 3,
});

const history = await classifier.fit(xTrain, yTrain, {
  validationData: [xVal, yVal],
  batchSize: BATCH_SIZE,
  epochs: NUM_EPOCHS,
  callbacks: [earlyStopping, restoreBestWeightsCallback],
});


Epoch 1 / 25


248ms 620us/step - acc=0.355 loss=1.36 val_acc=0.300 val_loss=1.32 
Epoch 2 / 25


175ms 438us/step - acc=0.605 loss=1.24 val_acc=0.520 val_loss=1.23 
Epoch 3 / 25


182ms 454us/step - acc=0.822 loss=1.09 val_acc=0.800 val_loss=1.11 
Epoch 4 / 25


230ms 574us/step - acc=0.907 loss=0.927 val_acc=0.830 val_loss=0.970 
Epoch 5 / 25


...

180ms 450us/step - acc=0.975 loss=0.363 val_acc=0.870 val_loss=0.573 
Epoch 9 / 25


179ms 447us/step - acc=0.982 loss=0.288 val_acc=0.890 val_loss=0.521 
Epoch 10 / 25


183ms 458us/step - acc=0.982 loss=0.232 val_acc=0.880 val_loss=0.481 
Epoch 11 / 25


184ms 459us/step - acc=0.990 loss=0.194 val_acc=0.880 val_loss=0.470 
Epoch 12 / 25


154ms 384us/step - acc=0.993 loss=0.166 val_acc=0.870 val_loss=0.454 


In [ ]:
console.log("Training complete.");
// log best weights
console.log("Best validation accuracy:", bestAcc);
// log best weights
if (bestWeights.length > 0) {
  console.log(
    "Best weights:",
    bestWeights.map((w) => w.dataSync())
  );
}


Training complete.
Best validation accuracy: 0.8899999856948853
Best weights: [
  Float32Array(589824) [
      -0.01458553597331047,   0.04647790640592575,  -0.11790028214454651,
       0.05183210223913193,  -0.04912508651614189,  -0.06355150789022446,
       -0.0712905302643776,  0.047946009784936905,  0.015527090057730675,
      0.029393872246146202,  0.011037350632250309,  -0.08336436748504639,
      -0.06574062258005142,  -0.04845399409532547,  -0.11720632761716843,
     -0.038035374134778976, -0.053958117961883545,   0.07522750645875931,
       -0.1012653335928917,  -0.10870110988616943,   0.04959658533334732,
       0.07256724685430527,  -0.03275788202881813, -0.006865565665066242,
      0.044840097427368164,  0.003523793537169695,  -0.03280274569988251,
      0.004993577022105455,  0.017164083197712898,   0.11629833281040192,
       -0.0934731662273407,  -0.08498488366603851,   0.00523048359900713,
       0.07440733909606934,   0.13781757652759552,   0.06484566628932953,
      0

## Evaluate model performance

Use tfjs `Model.evaluate` to get the loss and accuracy on the test dataset.



In [25]:
// @ts-expect-error expected Tensor<Rank> type
const evalResult = classifier.evaluate(xVal, yVal, {
  batchSize: 32,
});

console.log("Evaluation complete.");
const loss = evalResult[0].dataSync()[0];
const acc = evalResult[1].dataSync()[0];
console.log(`Test Loss: ${loss.toFixed(4)}`);
console.log(`Test Accuracy: ${acc.toFixed(4)}`);


Evaluation complete.
Test Loss: 0.5208
Test Accuracy: 0.8900


One way to evaluate your model performance is to visualize the classifier performance. Use `plotHistory` to see the loss and accuracy trends over the epochs.


In [ ]:
import { History } from "@tensorflow/tfjs-node";

function plotHistory(history: History) {
  const epochs = history.epoch;

  const lossTrace = {
    x: epochs,
    y: history.history.loss,
    type: "scatter",
    mode: "lines+markers",
    name: "Train Loss",
    xaxis: "x1",
    yaxis: "y1",
    line: { color: "#1f77b4" },
  };

  const valLossTrace = {
    x: epochs,
    y: history.history.val_loss,
    type: "scatter",
    mode: "lines+markers",
    name: "Validation Loss",
    xaxis: "x1",
    yaxis: "y1",
    line: { color: "#ff7f0e" },
  };

  const accTrace = {
    x: epochs,
    // eslint-disable-next-line @typescript-eslint/no-unnecessary-condition
    y: history.history.acc ?? history.history.accuracy,
    type: "scatter",
    mode: "lines+markers",
    name: "Train Accuracy",
    xaxis: "x2",
    yaxis: "y2",
    line: { color: "#2ca02c" },
  };

  const valAccTrace = {
    x: epochs,
    // eslint-disable-next-line @typescript-eslint/no-unnecessary-condition
    y: history.history.val_acc ?? history.history.val_accuracy,
    type: "scatter",
    mode: "lines+markers",
    name: "Validation Accuracy",
    xaxis: "x2",
    yaxis: "y2",
    line: { color: "#d62728" },
  };

  const html = `
  <div style="width: 100%; height: 600px;">
    <div id="loss-acc-plot" style="width: 100%; height: 100%;"></div>
    <script src="https://cdn.jsdelivr.net/npm/plotly.js-dist@latest/plotly.min.js"></script>
    <script>
      const data = ${JSON.stringify([lossTrace, valLossTrace, accTrace, valAccTrace])};

      const layout = {
        grid: { rows: 1, columns: 2, pattern: "independent" },
        height: 600,
        width: 1200,
        margin: { t: 60, l: 60, r: 40, b: 60 },
        annotations: [
          {
            text: "Loss", x: 0.225, y: 1.12, showarrow: false,
            font: { size: 18 }, xref: "paper", yref: "paper"
          },
          {
            text: "Accuracy", x: 0.775, y: 1.12, showarrow: false,
            font: { size: 18 }, xref: "paper", yref: "paper"
          }
        ],
        xaxis: { title: "Epoch", domain: [0, 0.45] },
        yaxis: { title: "Loss" },
        xaxis2: { title: "Epoch", domain: [0.55, 1] },
        yaxis2: { title: "Accuracy" },
        legend: {
          orientation: "h",
          y: -0.2,
          x: 0.5,
          xanchor: "center",
          font: { size: 12 }
        }
      };

      Plotly.newPlot("loss-acc-plot", data, layout, { responsive: true });
    </script>
  </div>
  `;

  tslab.display.html(html);
}

plotHistory(history);


Another way to view model performance, beyond just measuring loss and accuracy is to use a confusion matrix. The confusion matrix allows you to assess the performance of the classification model beyond accuracy. You can see what misclassified points get classified as. In order to build the confusion matrix for this multi-class classification problem, get the actual values in the test set and the predicted values.

Start by generating the predicted class for each example in the validation set using `Model.predict()`.

In [29]:
// @ts-expect-error expected Tensor<Rank> type
const yPredTensor = classifier.predict(xVal);
// @ts-expect-error expected Tensor<Rank> type
const yPredArray = Array.from(tf.argMax(yPredTensor, -1).dataSync());
const yValArray = Array.from(yVal.dataSync());


In [ ]:
function computeConfusionMatrix(labels: number[], predictions: number[], numClasses: number): number[][] {
  const matrix: number[][] = Array.from({ length: numClasses }, () => Array<number>(numClasses).fill(0));
  for (let i = 0; i < labels.length; i++) {
    const actual = labels[i];
    const predicted = predictions[i];
    matrix[actual][predicted]++;
  }
  return matrix;
}

const cm = computeConfusionMatrix(yValArray, yPredArray, dfTrainWithEmbeddings["Encoded Label"].unique().values.length);


In [33]:
console.log("Confusion Matrix (as DataFrame):");
const cmDf = new danfo.DataFrame(cm, {
  columns: dfTrainWithEmbeddings["Class Name"].unique().values,
  index: dfTrainWithEmbeddings["Class Name"].unique().values,
});
cmDf.print();


Confusion Matrix (as DataFrame):
╔════════════╤═══════════════════╤═══════════════════╤═══════════════════╤═══════════════════╗
║            │ sci.crypt         │ sci.electronics   │ sci.med           │ sci.space         ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ sci.crypt  │ 25                │ 0                 │ 0                 │ 0                 ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ sci.electr │ 5                 │ 20                │ 0                 │ 0                 ║
║ onics      │                   │                   │                   │                   ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ sci.med    │ 0                 │ 2                 │ 22                │ 1                 ║
╟────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────╢
║ sci.space  │ 2 

In [37]:
const classNames = dfTrainWithEmbeddings["Class Name"].unique().values as string[];

const html = `
<div style="width: 100%; height: 600px;">
  <div id="conf-matrix" style="width: 100%; height: 100%;"></div>
  <script src="https://cdn.jsdelivr.net/npm/plotly.js-dist@latest/plotly.min.js"></script>
  <script>
    const trace = {
      z: ${JSON.stringify(cm)},
      x: ${JSON.stringify(classNames)},
      y: ${JSON.stringify(classNames)},
      type: "heatmap",
      colorscale: "Blues",
      showscale: true,
      hoverongaps: false
    };

    const matrixLayout = {
      title: { text: "Confusion Matrix for Newsgroup Test Dataset", font: { size: 18 } },
      xaxis: { title: "Predicted Label", tickangle: -45 },
      yaxis: { title: "True Label" },
      height: 600,
      width: 700,
      margin: { t: 80, l: 100, r: 40, b: 100 }
    };

    Plotly.newPlot("conf-matrix", [trace], matrixLayout);
  </script>
</div>
`;

tslab.display.html(html);
